# Middleware
- To tightly control what happns inside the agent
    - Tracking agent behavior with logging, analytics, and debugging
    - Transforming prompts, tool selection, and output formatting
    - Adding retries, fallbacks, and early termination logic
    - Applying rate limits, guardrails, and PII detection
- How this can be achieved?
    - Hooks: We can have some wrappers before/after model calls or before/after tool calls to enforce some restrictions

In [1]:
import os
from dotenv import load_dotenv
load_dotenv()
from langchain.chat_models import init_chat_model
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")
model = init_chat_model("groq:qwen/qwen3-32b")
model

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.4.8', 'langchain': '1.3.11'}}, output_version=None, profile={'name': 'Qwen3 32B', 'release_date': '2024-12-23', 'last_updated': '2024-12-23', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 40960, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x000001E37B578050>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001E37B578D70>, model_name='qwen/qwen3-32b', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

## Builtin Middlewares - Some Examples
1. Summarization: Automatically summarize conversation history when approaching token limits.
2. Human in the Loop: Pause execution for human approval of tool calls.
3. Model Call Limit: Limit the number of model calls to prevent excessive costs.
4. Many more are here: https://docs.langchain.com/oss/python/langchain/middleware/built-in

### Summarization Middleware
#### With Message count as Trigger

In [2]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage, SystemMessage

agent = create_agent(
    model=model, # this is actual Model we are interacting
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model=model, # this can be small model which can do better summarization, here we are taking same model for an example
            # trigger=("tokens", 4000),
            trigger=("messages", 10), # when input + output length of no.of messages becomes 10 then it summarizes, we can have different triggers, please check documentation
            keep=("messages", 3), # It will keep top 3 messages after summarization
        ),
    ],
)

In [ ]:
# for demonstration purpose lets run different questions through threads(like different persons asking questions)
 #InMemorySaver expects this as it saves like a conversation, it needs which thread or user is invoking
config = {
    "configurable": {"thread_id": "test-1"}
}

In [ ]:
questions = {
    "What is 2+2=?",
    "What is (a+b)^2",
    "15-2=?",
    "What is 10**4",
    "what is y=mx+c? explain with example, only math no verbose answer",
    "what is 5^2=?"
}

for q in questions:
    response = agent.invoke(
        {
            "messages": [HumanMessage(content=q)]
        },
        config #InMemorySaver expects this as it saves like a conversation, it needs which thread or user is invoking
    )
    print("Response: ", response['messages'])
    print(f"Num of Message: {len(response['messages'])}")

Response:  [HumanMessage(content='What is 10**4', additional_kwargs={}, response_metadata={}, id='e1d525db-59c7-4d7c-817e-2990fc8016a1'), AIMessage(content="<think>\nOkay, so I need to figure out what 10 raised to the power of 4 is. Hmm, let me think. I remember that exponents mean multiplying the base number by itself a certain number of times. So 10 to the 4th power would be 10 multiplied by itself four times. Let me write that out to make sure.\n\nFirst, 10 multiplied by itself once is 10. Then, multiplying that by 10 again would be 10 x 10 = 100. Then, multiplying that result by another 10 would be 100 x 10 = 1,000. And one more time, multiplying by 10 again would be 1,000 x 10 = 10,000. So, 10^4 should be 10,000. \n\nWait, let me double-check that. Maybe there's a pattern here. I know that 10^1 is 10, 10^2 is 100, 10^3 is 1,000. Each time, adding another zero. So if that pattern continues, 10^4 would indeed be 10,000. That makes sense because each exponent just adds another zero. 

__Insight__: In the above scenario as per the result once the num of messages has reached 10, it has summarized the conversation and see next Num of messages started from 5 after 10, so Summarizer Middleware was triggered as per the config we have set

#### With Token Count as Trigger

In [ ]:
from langchain_core.tools import tool

@tool
def search_flights(source: str, destination: str) -> str:
    "Search Flights from the source to destination - Returns the long response to use more tokens"
    return f"""
    Flight options from {source} to {destination}
    1. Air India - Economy, with meals, with wifi costs 10000
    2. Spicejet - Economy, no meals, without wifi, with auto selected seats 8000
    3. Indigo - Business class, meals, wifi, infotainment, cab pickup and drop costs 35000
    """

agent = create_agent(
    model=model, # this is actual Model we are interacting
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model=model, # this can be small model which can do better summarization, here we are taking same model for an example
            trigger=("tokens", 3000), # when tokens reaches 3000 it will summarize
            keep=("messages", 200), # It will keep top 3 messages after summarization
        ),
    ],
)

config = {
    "configurable": {"thread_id": "test-1"}
}

# Approximate Token Counter logic
def count_tokens(messages):
    total_chars = sum(len(str(m.content)) for m in messages)
    return total_chars // 4 # 4 chars = 1 token

In [4]:
flights = [
    ("BLR", "HYD"),
    ("Hyd", "BLR"),
    ("Pune", "Delhi"),
    ("BLR", "Singapore"),
    ("Malaysia", "Hyderabad"),
    ("Hyderabad", "Dubai")
]

for source, destination in flights:
    response = agent.invoke(
        {
            "messages": [HumanMessage(content=f"Find Flights between {source} and {destination}")]
        },
        config #InMemorySaver expects this as it saves like a conversation, it needs which thread or user is invoking
    )

    tokens = count_tokens(response["messages"])
    print(f"{source} to {destination} tokens: {tokens}, {len(response['messages'])} messages")
    print(f"{(response['messages'])}")

BLR to HYD tokens: 936, 2 messages
[HumanMessage(content='Find Flights between BLR and HYD', additional_kwargs={}, response_metadata={}, id='7bc80bdd-531b-4629-96d7-61861c98d33d'), AIMessage(content="<think>\nOkay, the user is asking for flights between BLR and HYD. Let me start by recalling what BLR and HYD stand for. BLR is Bengaluru International Airport, and HYD is Hyderabad Rajiv Gandhi International Airport. So they're looking for flights between Bengaluru and Hyderabad in India.\n\nFirst, I should check if they have specific dates in mind. Since their query doesn't mention dates, I'll need to ask for that. But maybe they just want general information. Also, they might be looking for the cheapest, fastest, or most convenient options. Different airlines operate these routes, like IndiGo, Air India, SpiceJet, etc. The flight duration is usually around 1 hour and 15 minutes to 1 hour and 30 minutes, depending on the airline and any layovers.\n\nWait, are there direct flights between

__Insight__: In the above scenario as per the result once the num of tokens crossed 3000, i.e., after 3rd question it has summarizd, copy the above output and search for summary we will find the summary, it has summarized the conversation as per the configuration set

### Human in the Loop Middleware
Pause execution for human approval of tool calls.

In [22]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage, SystemMessage

In [23]:
from langchain.chat_models import init_chat_model
os.environ["GOOGLE_API_KEY"] = os.getenv("GOOGLE_API_KEY")
model = init_chat_model("google_genai:gemini-2.5-flash")

In [24]:
def read_email_tool(email_id: str) -> str:
    """Mock function to read an email by its Id"""
    return f"Email content for ID: {email_id}"

def send_email_tool(recipient: str, subject: str, body: str) -> str:
    """Mock function to send an email"""
    return f"""Email sent to {recipient} with subject: {subject}"""

In [38]:
agent = create_agent(
    model = model, # this is actual Model we are interacting
    checkpointer = InMemorySaver(),
    tools = [read_email_tool, send_email_tool],
    middleware = [
            HumanInTheLoopMiddleware(
                interrupt_on={
                    "send_email_tool": {
                        "allowed_decisions": ['approve', 'edit', 'reject']
                    },
                    "read_email_tool": False # no interruption for this tool
                }
            )
        ]
)

In [39]:
config = {
    "configurable": {"thread_id": "test-approve"}
}

result = agent.invoke(
    {"messages": [
        HumanMessage(content="""Send email to xyz@gmail.com with subject "test message" and body "test body msg" """)
    ]},
    config = config
)

In [47]:
result['__interrupt__']

[Interrupt(value={'action_requests': [{'name': 'send_email_tool', 'args': {'subject': 'test message', 'body': 'test body msg', 'recipient': 'xyz@gmail.com'}, 'description': "Tool execution requires approval\n\nTool: send_email_tool\nArgs: {'subject': 'test message', 'body': 'test body msg', 'recipient': 'xyz@gmail.com'}"}], 'review_configs': [{'action_name': 'send_email_tool', 'allowed_decisions': ['approve', 'edit', 'reject']}]}, id='3be07d231d6c582b69e0ea3021c97c77')]

__Insight__: In the above result it clearly shows the `__interrupt__` respnse, where we need to either respond with approve, reject or edit, below is the functionality to do the same

#### Approve action through Human in the Loop

In [ ]:
from langgraph.types import Command
if("__interrupt__" in result):
    print("Paused! Approving....")

    result = agent.invoke(
        Command(
            resume={
                "decisions": [
                    {"type": "approve"} # we are approving the send email action, so that LLM can proceed with the action
                ]
            }
        ),
        config=config
    )
    print(result['messages'][-1].content)

Paused! Approving....
I have sent an email to xyz@gmail.com with the subject "test message" and body "test body msg".


#### Edit Action through Human in the Loop

In [49]:
from langgraph.types import Command

config = {
    "configurable": {"thread_id": "test-approve"}
}

result = agent.invoke(
    {"messages": [
        HumanMessage(content="""Send email to xyz@gmail.com with subject "test message" and body "test body msg" """)
    ]},
    config = config
)

if("__interrupt__" in result):
    print("Paused! Editing....")

    result = agent.invoke(
        Command(
            resume={
                "decisions": [
                    {
                        "type": "edit",
                        "edited_action": {
                            "name": "send_email_tool",
                            "args": {
                                "recipient": "abc@gmail.com",
                                "subject": "hello",
                                "body": "Edited Test message"
                            }
                        }
                    }
                ]
            }
        ),
        config=config
    )
    print(result['messages'][-1].content)

Paused! Editing....
I have sent an email to abc@gmail.com with the subject "hello" and body "Edited Test message".


`Production Best Practices`
1. Use a DB Checkpointer: Replace InMemorySaver with PostgresSaver or RedisSaver. Memory saver will lose the agent's state if the backend server restarts or restarts/autoscales.
2. Handle Expirations/Timeouts: Humans might take hours or days to approve. Since the state is in the database, the agent isn't consuming CPU/RAM while waiting. However, you should implement cleanup jobs to expire stale approval requests.
3. Verify Security/Authorization: Do not trust user inputs blindly. Before allowing someone to resume a thread, verify that their user token is authorized to access the given thread_id and has permissions to approve that specific action.